<a href="https://colab.research.google.com/github/Ali-Shahrez/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone -q https://github.com/Ali-Shahrez/flyrank-ml-internship.git
%cd flyrank-ml-internship

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows")

/content/flyrank-ml-internship
Loaded 30,000 rows


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 2 (Refresh/Content Opportunity Scoring) is a **ranking/scoring** task, not
plain classification. The decision it supports is "which pages should a content
strategist review first?" — that's a "which ones first?" question, and per the
framing-ml-problems skill, that maps to ranking/scoring, with precision@K as the
natural metric. A binary decline/not-decline classifier alone wouldn't answer the
real question, since a strategist has limited review capacity and needs an ordered
list, not just a yes/no per page.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter dataset's target is `is_declining_label = (trend_direction == "down")`.

**This label comes from a defined rule, not an observed outcome.** `trend_direction`
is computed by thresholding `trend_pct` (last-30-day impressions vs. the prior
30 days) at ±20% — a same-window calculation, not something measured to actually
happen afterward. That makes it a **proxy label**: useful for building and testing
a pipeline today, but it tells us what already happened in a short window, not
what will happen next. Per the flyrank-data skill, `trend_direction` and `trend_pct`
are label sources only — never features — since using them as inputs would let
the model just learn to reproduce the rule that created the label (a circular result).

A stronger version of this target would be an **observed future outcome** —
e.g. "declines over the next 30 days," measured after a clear decision point —
which is the direction a stronger capstone should move toward.

In [3]:
print(df['trend_direction'].value_counts())
print(f"\nShare labeled 'down': {(df['trend_direction']=='down').mean():.1%}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Share labeled 'down': 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** — of the top 50 pages the model ranks highest, how many are
actually labeled declining? This matters more than raw accuracy for a ranking
task, but it must be read against the base rate, not in isolation: 54.2% of
pages in this dataset are labeled declining, so random guessing would already
score ~0.542 on this metric. Measured against that baseline, the documented
starter benchmarks (client-holdout validated, from `outputs/model_report.md`)
tell a sharper story: baseline hand rule 0.240 (worse than random), decision
tree 0.540 (roughly equal to random), random forest 0.740 (clearly better than
random). A "good" Precision@50 here isn't just a high number — it's a number
meaningfully above 0.542.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Shape: {df.shape}")
print(f"One row = one pseudonymized content item (a single page), belonging to one client.")
df.head(3)

Shape: (30000, 44)
One row = one pseudonymized content item (a single page), belonging to one client.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


One row = one content item (a page) for one client, with trailing-90-day
metrics attached. 30,000 rows, 44 columns, spanning 32 clients.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule was actually tried first (`hand_rule_score`), and it looked strong
in-sample (Precision@50: 0.680) — but on a true client-holdout test, it dropped
to 0.420. Note the test set's own base rate here is 39.1% (not the dataset-wide
54.2% — client-level holdout splits shift the base rate depending on which
clients land in the test set), so 0.420 is only modestly above chance, while
the decision tree (0.560-0.600) is more clearly and consistently ahead of it.
This shows two things at once: a hand-tuned rule can overfit to the clients it
was built around, and any Precision@K number needs to be checked against its
own test set's base rate before being called "good" — not just compared across
methods in isolation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.